# Pharmasense — ST-GNN Training
**Spatio-Temporal Graph Neural Network for communicable disease outbreak prediction**

Architecture: GraphSAGE (spatial) + GRU (temporal) → sigmoid risk score per district per disease

**Runtime:** GPU (T4). Go to Runtime → Change runtime type → T4 GPU before running.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/01_train_stgnn.ipynb)

## 0. Install dependencies

In [ ]:
import subprocess, sys

# Install torch first (CUDA 11.8 for Colab T4)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)

# PyG and dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric',
    'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
    '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu118.html'], check=True)

# Other deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'psycopg2-binary', 'pandas', 'numpy', 'scikit-learn', 'tqdm', 'matplotlib'], check=True)

print('All dependencies installed.')

## 1. Config — fill in your Supabase DB password

In [ ]:
# ── Supabase PostgreSQL connection ────────────────────────────────────────────
DB_HOST     = 'db.xwtvujwdybvjbaucshbt.supabase.co'
DB_PORT     = 5432
DB_NAME     = 'postgres'
DB_USER     = 'postgres'
DB_PASSWORD = 'pharmasons99lite'   # <-- your DB password

# ── Training config ───────────────────────────────────────────────────────────
DISEASES        = ['dengue', 'malaria', 'cholera', 'acute_diarrhoeal_disease']
LOOKBACK_WEEKS  = 12    # use last 12 weeks to predict
HORIZON_WEEKS   = 2     # predict 2 weeks ahead (14 days)
HIDDEN_DIM      = 64
NUM_LAYERS      = 2
GRU_LAYERS      = 2
DROPOUT         = 0.3
LEARNING_RATE   = 1e-3
EPOCHS          = 100
BATCH_SIZE      = 32    # number of districts per mini-batch
RISK_THRESHOLD  = 75    # cases_per_100k above which = positive class
TRAIN_SPLIT     = 0.75
VAL_SPLIT       = 0.15  # remaining 0.10 = test

# ── Google Drive output path (where to save the .pt model) ───────────────────
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Training will be slow.')

## 2. Mount Google Drive (for saving model artifacts)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive mounted. Artifacts will be saved to: {DRIVE_OUTPUT}')

## 3. Load data from Supabase

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import json

print('Connecting to Supabase...')
conn = psycopg2.connect(
    host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
    user=DB_USER, password=DB_PASSWORD, sslmode='require'
)
print('Connected.')

# Load districts + adjacency
districts_df = pd.read_sql("""
    SELECT id, name, state, centroid_lat, centroid_lon, adjacency_ids
    FROM districts ORDER BY id;
""", conn)
print(f'Districts: {len(districts_df)}')

# Load epidemiological data
epid_df = pd.read_sql("""
    SELECT district_id, disease, iso_week, week_start_date, cases, deaths
    FROM epid_weekly
    ORDER BY week_start_date, district_id, disease;
""", conn)
conn.close()
print(f'Epid records: {len(epid_df)}')
print(f'Date range: {epid_df["week_start_date"].min()} -> {epid_df["week_start_date"].max()}')
epid_df['week_start_date'] = pd.to_datetime(epid_df['week_start_date'])
print('\nCases by disease:')
print(epid_df.groupby('disease')['cases'].sum())

## 4. Build graph + feature tensors

In [ ]:
from sklearn.preprocessing import StandardScaler

# ── District index ────────────────────────────────────────────────────────────
dist_ids   = list(districts_df['id'])
id_to_idx  = {id_: i for i, id_ in enumerate(dist_ids)}
N          = len(dist_ids)  # number of nodes

# ── Build edge_index from adjacency_ids ───────────────────────────────────────
src_list, dst_list = [], []
for _, row in districts_df.iterrows():
    i = id_to_idx[row['id']]
    neighbors = row['adjacency_ids'] or []
    for nbr_id in neighbors:
        j = id_to_idx.get(nbr_id)
        if j is not None:
            src_list.append(i)
            dst_list.append(j)

edge_index = torch.tensor([src_list, dst_list], dtype=torch.long, device=DEVICE)
print(f'Graph: {N} nodes, {edge_index.shape[1]} edges')

# ── Build time series: weeks × nodes × features ───────────────────────────────
# Features per node per week: [cases_dengue, cases_malaria, cases_cholera, cases_add, temp_kelvin, preci, lai]
# For now: 4 disease case counts (normalized). Weather added in Phase 2 inference.

all_weeks = sorted(epid_df['week_start_date'].unique())
T = len(all_weeks)  # total time steps
week_to_idx = {w: i for i, w in enumerate(all_weeks)}

# Initialize feature matrix: [T x N x D_in]
D_in = len(DISEASES)  # 4 disease channels
X = np.zeros((T, N, D_in), dtype=np.float32)

for _, row in epid_df.iterrows():
    t = week_to_idx.get(row['week_start_date'])
    n = id_to_idx.get(row['district_id'])
    d = DISEASES.index(row['disease']) if row['disease'] in DISEASES else None
    if t is not None and n is not None and d is not None:
        X[t, n, d] += row['cases']

# Normalize per-disease channel with log1p + StandardScaler
X = np.log1p(X)
scalers = []
for d in range(D_in):
    sc = StandardScaler()
    flat = X[:, :, d].reshape(-1, 1)
    X[:, :, d] = sc.fit_transform(flat).reshape(T, N)
    scalers.append(sc)

print(f'Feature matrix shape: {X.shape}  (T={T} weeks, N={N} nodes, D={D_in} features)')

# ── Build labels: is the district in HIGH risk next HORIZON_WEEKS? ──────────────
# Label = 1 if cases in next horizon > RISK_THRESHOLD cumulative
Y = np.zeros((T, N, len(DISEASES)), dtype=np.float32)  # regression: normalized future cases
for t in range(T - HORIZON_WEEKS):
    Y[t] = X[t + HORIZON_WEEKS]  # predict the normalized future state

print(f'Label matrix shape: {Y.shape}')

## 5. Define ST-GNN model (GraphSAGE + GRU)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class STGNN(nn.Module):
    """
    Spatio-Temporal GNN for outbreak prediction.
    
    Architecture:
      1. GraphSAGE layers: spatial message passing across district adjacency
      2. GRU: temporal modelling across the lookback window
      3. Linear head: predict normalized case counts per disease
    
    Input:  [batch_size, lookback, N, D_in]
    Output: [batch_size, N, D_out]  (D_out = D_in = num_diseases)
    """
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_sage_layers=2, num_gru_layers=2, dropout=0.3):
        super().__init__()
        self.in_channels  = in_channels
        self.hidden       = hidden_channels
        self.dropout      = dropout

        # ── GraphSAGE spatial layers ──
        self.sage_layers = nn.ModuleList()
        self.sage_layers.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_sage_layers - 1):
            self.sage_layers.append(SAGEConv(hidden_channels, hidden_channels))

        # ── GRU temporal layer ──
        # Input: hidden_channels per node per timestep → GRU over time
        self.gru = nn.GRU(
            input_size=hidden_channels,
            hidden_size=hidden_channels,
            num_layers=num_gru_layers,
            batch_first=True,
            dropout=dropout if num_gru_layers > 1 else 0.0
        )

        # ── Output head ──
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, x_seq, edge_index):
        """
        x_seq:      [T, N, D_in] — sequence of node features
        edge_index: [2, E]       — graph connectivity
        Returns:    [N, D_out]   — predictions for next horizon
        """
        T, N, _ = x_seq.shape
        sage_outputs = []

        # Apply GraphSAGE at each timestep
        for t in range(T):
            h = x_seq[t]  # [N, D_in]
            for layer in self.sage_layers:
                h = layer(h, edge_index)
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)
            sage_outputs.append(h)  # [N, hidden]

        # Stack: [N, T, hidden] → run GRU over time dimension
        sage_seq = torch.stack(sage_outputs, dim=1)  # [N, T, hidden]
        gru_out, _ = self.gru(sage_seq)              # [N, T, hidden]
        last_out   = gru_out[:, -1, :]               # [N, hidden] — last timestep

        return self.head(last_out)  # [N, D_out]


model = STGNN(
    in_channels=D_in,
    hidden_channels=HIDDEN_DIM,
    out_channels=len(DISEASES),
    num_sage_layers=NUM_LAYERS,
    num_gru_layers=GRU_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: STGNN')
print(f'Params: {total_params:,}')
print(model)

## 6. Train / Val / Test split + training loop

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# ── Sliding window dataset ────────────────────────────────────────────────────
# Each sample: X[t:t+LOOKBACK], Y[t+LOOKBACK]
samples = []
T_valid = T - LOOKBACK_WEEKS - HORIZON_WEEKS
for t in range(T_valid):
    x_window = X[t:t + LOOKBACK_WEEKS]          # [LOOKBACK, N, D]
    y_target  = Y[t + LOOKBACK_WEEKS - 1]       # [N, D] — label at end of window
    samples.append((x_window, y_target))

n_train = int(len(samples) * TRAIN_SPLIT)
n_val   = int(len(samples) * VAL_SPLIT)
train_samples = samples[:n_train]
val_samples   = samples[n_train:n_train + n_val]
test_samples  = samples[n_train + n_val:]

print(f'Samples — Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

# ── Training ──────────────────────────────────────────────────────────────────
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5, verbose=True)
criterion = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0
EARLY_STOP    = 20

edge_index_dev = edge_index.to(DEVICE)

for epoch in tqdm(range(EPOCHS), desc='Training'):
    # ── Train ──
    model.train()
    epoch_loss = 0.0
    np.random.shuffle(train_samples)
    for x_np, y_np in train_samples:
        x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)  # [T, N, D]
        y = torch.tensor(y_np, dtype=torch.float32, device=DEVICE)  # [N, D]
        optimizer.zero_grad()
        pred = model(x, edge_index_dev)  # [N, D]
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    avg_train = epoch_loss / len(train_samples)
    train_losses.append(avg_train)

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_np, y_np in val_samples:
            x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
            y = torch.tensor(y_np, dtype=torch.float32, device=DEVICE)
            pred = model(x, edge_index_dev)
            val_loss += criterion(pred, y).item()
    avg_val = val_loss / len(val_samples)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= EARLY_STOP:
            print(f'Early stopping at epoch {epoch+1}')
            break

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | train={avg_train:.4f} | val={avg_val:.4f} | best_val={best_val_loss:.4f}')

# ── Plot loss curves ──
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('STGNN Training')
plt.legend(); plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'stgnn_loss_curve.png')
plt.show()
print(f'Best val loss: {best_val_loss:.4f}')

## 7. Evaluate on test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.load_state_dict(best_state)
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for x_np, y_np in test_samples:
        x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
        pred = model(x, edge_index_dev).cpu().numpy()
        all_preds.append(pred)
        all_targets.append(y_np)

preds   = np.concatenate(all_preds,   axis=0)  # [n_test*N, D]
targets = np.concatenate(all_targets, axis=0)

print('Test Set Metrics (normalized space):')
print(f'  MAE:  {mean_absolute_error(targets, preds):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(targets, preds)):.4f}')
print(f'  R2:   {r2_score(targets.reshape(-1), preds.reshape(-1)):.4f}')

print('\nPer-disease metrics:')
for i, disease in enumerate(DISEASES):
    mae  = mean_absolute_error(targets[:, i], preds[:, i])
    rmse = np.sqrt(mean_squared_error(targets[:, i], preds[:, i]))
    r2   = r2_score(targets[:, i], preds[:, i])
    print(f'  {disease:30s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}')

## 8. Save model artifact

In [ ]:
import json
from datetime import datetime

artifact = {
    'model_state_dict': best_state,
    'model_config': {
        'in_channels':      D_in,
        'hidden_channels':  HIDDEN_DIM,
        'out_channels':     len(DISEASES),
        'num_sage_layers':  NUM_LAYERS,
        'num_gru_layers':   GRU_LAYERS,
        'dropout':          DROPOUT,
    },
    'training_config': {
        'lookback_weeks':   LOOKBACK_WEEKS,
        'horizon_weeks':    HORIZON_WEEKS,
        'diseases':         DISEASES,
        'risk_threshold':   RISK_THRESHOLD,
    },
    'district_index': {
        'id_to_idx':   id_to_idx,
        'n_districts': N,
    },
    'scalers':    scalers,   # for inverse_transform at inference time
    'val_loss':   best_val_loss,
    'trained_at': datetime.now().isoformat(),
    'data_range': {
        'start': str(all_weeks[0]),
        'end':   str(all_weeks[-1]),
        'n_weeks': T,
    }
}

save_path = DRIVE_OUTPUT + 'stgnn_latest.pt'
torch.save(artifact, save_path)
print(f'Model saved to: {save_path}')

# Download to local machine
from google.colab import files
files.download(save_path)
print('Download started — save to ml/artifacts/stgnn_latest.pt in your local repo')